# Modeling — XGBoost (quantile 0.9)

Desain: `docs/superpowers/specs/2026-08-19-xgboost-modeling-design.md`.
Rencana: `docs/superpowers/plans/2026-08-19-xgboost-modeling.md`.

Notebook ini tipis dengan sengaja. Semua logika ada di `utils/walk_forward.py`,
`utils/model_common.py` dan `utils/model_xgboost.py`, supaya jalur skrip dan
jalur notebook tidak bisa berbeda.

**Desember 2025 terkunci** dan tidak dinilai di sini.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from utils.modelling import evaluation, model_xgboost as xgb
from utils.modelling import modeling_prep, run_config, walk_forward

# Di mana boosting dijalankan. "cpu" adalah default dan satu-satunya
# angka yang pernah diukur sejauh ini; "cuda" untuk mesin sewaan.
# Grid 19 titik membangun 19 pohon per ronde boosting (T-14), jadi
# tahap ini yang paling mahal di Fase 3.
#
# SATU MODEL = SATU DEVICE, dari benchmark sampai fit final. Kandidat
# yang dipilih di satu device lalu difit ulang di device lain tidak
# dipilih di bawah aritmetika yang menghasilkannya — hist di GPU dan
# hist di CPU tidak membangun pohon yang identik. Device tercatat di
# bundle supaya itu terbaca, bukan diasumsikan.
#
# Keempat setelan di bawah dibaca dari environment; tanpa satu pun di
# antaranya, notebook ini berperilaku persis seperti sebelum jalur cloud
# ada — termasuk nama berkas yang ditulisnya. Untuk memecah pencarian:
#
#   FORECAST_SHARD=0-14 FORECAST_DEVICE=cuda:0 \
#   FORECAST_MODEL_INPUT=/kaggle/input/forecast-scm/model_input.parquet \
#   FORECAST_CHECKPOINT_DIR=/kaggle/working jupyter nbconvert ...
#
# Rencana mesin per tahap ada di
# docs/superpowers/specs/2026-08-24-distributed-gpu-training-design.md.
DEVICE = run_config.device(xgb.DEFAULT_DEVICE)
SHARD = run_config.shard()
SEARCH_FILE = run_config.search_checkpoint("xgb")
RESULTS_FILE = run_config.checkpoint_path("xgb_walk_forward_results.csv")

df = pd.read_parquet(run_config.model_input_path())
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(run_config.describe(DEVICE))


## Benchmark

Satu putaran dua-fit di fold 5 dengan `DEFAULT_PARAMS`, untuk mengukur ongkos
sebelum 60 fit pencarian dijalankan dan melihat di ronde berapa early stopping
mendarat. Angkanya dicatat di `docs/hasil-modeling-xgb.md`.

In [ ]:
import resource
import time

split = walk_forward.prepare_fold(df, 5)
train, valid = split["train"], split["valid"]
fit_rows, es_rows = xgb.split_early_stopping(train)
print(f"train {len(train):,} rows -> fit {len(fit_rows):,} + tail {len(es_rows):,}")
print(f"valid {len(valid):,} rows")
print(f"QUANTILE_SET: {len(xgb.QUANTILES)} titik, {xgb.QUANTILES[0]}..{xgb.QUANTILES[-1]}")

fit_predict = xgb.make_fit_predict(dict(xgb.DEFAULT_PARAMS), device=DEVICE)
start = time.time()
prediction = fit_predict(train, valid)
elapsed = time.time() - start

peak_bytes = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss  # bytes on macOS
headline = min(range(len(xgb.QUANTILES)),
               key=lambda i: abs(xgb.QUANTILES[i] - evaluation.DEFAULT_ALPHA))
print(f"best_iteration {fit_predict.best_iterations[0]} of {xgb.MAX_ROUNDS}")
print(f"wall time {elapsed / 60:.1f} min (both fits)")
print(f"peak RSS  {peak_bytes / 1024 ** 3:.2f} GB")
print(f"prediction shape {prediction.shape} (baris x titik kuantil)")
print(f"di tau=0.9: mean {prediction[:, headline].mean():.2f}, "
      f"max {prediction[:, headline].max():.2f}")
# Head komposit tidak punya jaminan monotonisitas struktural: angka ini
# diukur, bukan diasumsikan nol, dan di atas beberapa persen ia sinyal bahwa
# arctan pinball loss (Sluijterman dkk. 2024) sepadan dengan kerumitannya.
print(f"crossing_rate {evaluation.crossing_rate(prediction, xgb.QUANTILES):.4f}")


## Pencarian hyperparameter

30 kandidat dari ruang 2.592 kombinasi, dinilai di fold 3 dan 5 dengan
pinball@0.9 gabungan. Hasil yang dilaporkan datang dari walk-forward lima fold
di bawah, bukan dari sini — menilai di fold yang memilih pemenang akan
optimistis.

In [ ]:
# Ke-30 kandidat dijalankan ulang: objective-nya kini quantile_alpha =
# seluruh QUANTILE_SET, jadi skor lama (pinball@0,9) bukan K1 dan tidak
# sebanding. Artefak run kuantil-tunggal sudah diganti nama menjadi
# `*.single-quantile.bak.*` (2026-08-24) sesudah guard checkpoint
# diverifikasi berbunyi, jadi sel ini mulai dari nol. Kalau berkas tanpa
# kolom `headline_quantile` muncul lagi di jalur checkpoint, guard yang
# sama akan menolaknya dalam hitungan detik — itu perilaku yang diinginkan.
#
# Dengan FORECAST_SHARD diset, mesin ini hanya menjalankan kandidat yang
# jadi bagiannya, tetapi penomorannya tetap absolut terhadap seed 42 —
# itulah yang membuat dua shard bisa disatukan `model_common.merge_shards()`
# nanti. Kolom `device` dan `commit` ikut ditulis ke tiap baris supaya
# angkanya dapat ditelusuri ke mesin yang melahirkannya.
candidates = xgb.sample_search_space(xgb.N_CANDIDATES, seed=42)
search_results = xgb.run_search(df, candidates, folds=xgb.SEARCH_FOLDS,
                                checkpoint_path=SEARCH_FILE, device=DEVICE,
                                only=SHARD,
                                provenance=run_config.provenance(DEVICE))
search_results.to_csv(SEARCH_FILE, index=False)
# `pinball` di sini adalah K1. Kolom *_headline dibaca di tau=0,9.
search_results.sort_values("pinball").head(10)


## Walk-forward final

Konfigurasi pemenang di kelima fold, melawan ketiga baseline naive pada baris
yang identik.

In [ ]:
# Berhenti di sini kalau ini run bershard: `select_best()` di bawah akan
# memilih pemenang dari sebagian kandidat saja, dan hasilnya akan tampak
# sepenuhnya wajar. Gabungkan dulu seluruh shard di satu mesin —
#
#   from utils.modelling import model_common
#   merged = model_common.merge_shards(
#       ["xgb_search_results.shard-0-14.csv",
#        "xgb_search_results.shard-15-29.csv"],
#       candidates, xgb.SEARCH_SPACE)
#
# — lalu jalankan sel ini di mesin itu, tanpa FORECAST_SHARD.
#
# Walk-forward dan fit final ketiga model dijalankan di **satu mesin yang
# sama** (Mac lokal), karena wall time-nya masuk K3 dan K3-lah yang
# menentukan pemenang saat K1 seri — lihat Bagian 1 spec eksekusi
# terdistribusi. Ia juga tidak punya checkpoint: sesi yang terpotong di
# tengah kehilangan seluruhnya.
assert SHARD is None, (
    "run bershard: jangan pilih pemenang dari sebagian kandidat — "
    "gabungkan seluruh shard dengan model_common.merge_shards() lebih dulu"
)

best = xgb.select_best(search_results, candidates)
xgb.save_best_params(best)
print(best)

fit_predict = xgb.make_fit_predict(best, device=DEVICE)
results = walk_forward.run_walk_forward(df, fit_predict, model_name="xgboost",
                                        quantiles=xgb.QUANTILES)
results.to_csv(RESULTS_FILE, index=False)

print("best_iteration per fold:", fit_predict.best_iterations)
print(f"K1 (rata-rata pinball lintas {len(xgb.QUANTILES)} kuantil): "
      f"{walk_forward.pooled_k1(results, 'xgboost'):.4f}")

overall = results[results["group_col"].isna()]
(overall[(overall["quantile"] - evaluation.DEFAULT_ALPHA).abs() < 1e-9]
 .pivot_table(index="model", columns="fold_id", values="pinball").round(3))


In [ ]:
bundle = xgb.fit_final(df, best, device=DEVICE)
xgb.save_bundle(bundle)
print(f"trained on {bundle['n_train']:,} rows, "
      f"{len(bundle['columns'])} columns, encoding {bundle['encoding']}, "
      f"{bundle['best_iteration']} rounds, device {bundle['device']}, "
      f"{len(bundle['quantiles'])} titik kuantil "
      f"({bundle['quantiles'][0]}..{bundle['quantiles'][-1]})")


## Hasil

Tiga potongan, masing-masing melawan ketiga baseline naive pada baris identik.
Satu angka global menyesatkan di data yang 44% targetnya nol.

In [ ]:
results = pd.read_csv(xgb.RESULTS_FILE)
HEADLINE = evaluation.DEFAULT_ALPHA

print("=== K1 (rata-rata pinball lintas QUANTILE_SET, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} K1 {walk_forward.pooled_k1(results, model):7.4f}")

print("\n=== per fold, K1 ===")
print(results[results["group_col"].isna()]
      .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

print(f"\n=== per fold, pinball di tau={HEADLINE} (angka headline B-9) ===")
headline_rows = results[results["group_col"].isna()
                        & ((results["quantile"] - HEADLINE).abs() < 1e-9)]
print(headline_rows.pivot_table(index="model", columns="fold_id",
                                values="pinball").round(3))

for group_col in walk_forward.GROUP_COLS:
    print(f"\n=== per {group_col} (K1, pooled over folds) ===")
    grouped = results[results["group_col"] == group_col]
    # Kolom dipilih sebelum apply: tanpa itu pandas ikut menyertakan kolom
    # pengelompokan dan mengeluarkan FutureWarning di setiap sel.
    table = (grouped.assign(weighted=grouped["pinball"] * grouped["n"])
                    .groupby(["model", "group_value"], observed=True)[["weighted", "n"]]
                    .apply(lambda part: part["weighted"].sum() / part["n"].sum())
                    .unstack())
    print(table.round(3))

# pooled_metric menolak dirata-ratakan lintas kuantil untuk metrik selain
# pinball/crossing_rate — coverage di 0,05 dan di 0,95 menjawab pertanyaan
# yang berbeda. Jadi ketiganya dibaca di tau headline, eksplisit.
print(f"\n=== coverage / fill rate di tau={HEADLINE} (overall, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} "
          f"coverage {walk_forward.pooled_metric(results, model, 'coverage', quantile=HEADLINE):6.3f}  "
          f"fill_rate {walk_forward.pooled_metric(results, model, 'fill_rate', quantile=HEADLINE):6.3f}  "
          f"shortfall {walk_forward.pooled_metric(results, model, 'shortfall_units', quantile=HEADLINE):9.1f}  "
          f"crossing {walk_forward.pooled_metric(results, model, 'crossing_rate'):6.4f}")

print("\n=== K2: coverage per titik kuantil (xgboost) ===")
print(walk_forward.coverage_by_quantile(results, "xgboost").round(4)
      .to_string(index=False))


## Head-to-head lawan Random Forest

Sah dilakukan karena kedua model dinilai di baris yang identik — dijamin
`walk_forward.eligible_rows()`, bukan oleh disiplin. Fold 1, 2, 4 adalah
potongan yang bersih: keduanya memilih pemenang di fold 3 dan 5.

In [ ]:
from utils.modelling import model_random_forest as rf

rf_results = pd.read_csv(rf.RESULTS_FILE)
combined = pd.concat([results, rf_results], ignore_index=True)
HEADLINE = evaluation.DEFAULT_ALPHA

for label, folds in (("semua fold", None), ("fold 1/2/4 (bersih)", (1, 2, 4))):
    print(f"=== {label} ===")
    for model in ("xgboost", "random_forest", "naive_roll_mean_7"):
        rows = combined[(combined["model"] == model) & combined["group_col"].isna()]
        if rows.empty:
            continue
        print(f"{model:20s} "
              f"K1 {walk_forward.pooled_k1(combined, model, folds):6.3f}  "
              f"mae@0.9 {walk_forward.pooled_metric(combined, model, 'mae', folds, quantile=HEADLINE):7.3f}  "
              f"coverage@0.9 {walk_forward.pooled_metric(combined, model, 'coverage', folds, quantile=HEADLINE):6.3f}")
    print()

# Lantai naif dinilai di seluruh 19 titik juga — ia ramalan titik, jadi ia
# membayar penalti pinball karena berpura-pura satu angkanya adalah setiap
# kuantil sekaligus. K1 baseline karenanya BUKAN angka 6,56 pinball@0,9 yang
# tercatat di dokumen hasil lama; keduanya tidak boleh disandingkan (T-10).
